# Entrenamiento de CNN con Transfer Learning
## Objetivos
- un baseline simple con transfer learning
- congelar inicialmente el backbone,
- entrenar solo la cabeza clasificadora,
- luego hacer fine-tuning parcial o total,
- guardar métricas por época,
- guardar el mejor modelo,
- matriz de confusión,
- accuracy, precision, recall y F1.

In [2]:
# 1. Configuración inicial
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from torchvision import datasets
from torchvision import transforms as tfs

from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from _bootstrap import setup
from src.vasijas.reporting import ProcessingReport

PATHS = setup()
report = ProcessingReport()

# Configuración general
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS = 10
LR = 1e-3

DATA_DIR = "../../data/splits"  # train/val/test
MODEL_NAME = "resnet18"

print("Device:", DEVICE)

Device: cuda


In [ ]:
# 2. Transformaciones y DataLoaders
train_transforms = tfs.Compose([
    tfs.Resize((IMG_SIZE, IMG_SIZE)),
    tfs.RandomHorizontalFlip(),
    tfs.RandomRotation(10),
    tfs.ToTensor(),
    tfs.Normalize(mean=[0.5], std=[0.5])

])

val_transforms = tfs.Compose([
    tfs.Resize((IMG_SIZE, IMG_SIZE)),
    tfs.ToTensor(),
    tfs.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_dataset.classes
NUM_CLASSES = len(class_names)

print("Clases:", class_names)

In [ ]:
# 3. Visualización rápida (sanity check)
def show_batch(loader):
    images, labels = next(iter(loader))
    images = images[:8]
    labels = labels[:8]

    fig, axes = plt.subplots(1, 8, figsize=(16, 3))
    for i, ax in enumerate(axes):
        img = images[i].permute(1, 2, 0).numpy()
        ax.imshow(img.squeeze(), cmap="gray")
        ax.set_title(class_names[labels[i]])
        ax.axis("off")


show_batch(train_loader)

In [ ]:
# 4. Definición del modelo (ResNet / VGG)
def get_model(model_name, num_classes):
    if model_name == "resnet18":
        model = models.resnet18(weights="IMAGENET1K_V1")
        for param in model.parameters():
            param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "vgg16":
        model = models.vgg16(weights="IMAGENET1K_V1")
        for param in model.parameters():
            param.requires_grad = False
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

    else:
        raise ValueError("Modelo no soportado")

    return model.to(DEVICE)


model = get_model(MODEL_NAME, NUM_CLASSES)

In [ ]:
# 5. Funciones de entrenamiento y validación
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc


def validate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc, all_preds, all_labels

In [ ]:
# 6. Loop de entrenamiento
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, preds, labels = validate(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
# 7. Curvas de entrenamiento
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss")
plt.show()

plt.plot(train_accs, label="Train Acc")
plt.plot(val_accs, label="Val Acc")
plt.legend()
plt.title("Accuracy")
plt.show()

In [ ]:
# 8. Evaluación final
print(classification_report(labels, preds, target_names=class_names))

cm = confusion_matrix(labels, preds)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# 9. Guardado del modelo
torch.save(model.state_dict(), PATHS.OUTPUT_MODELS / f"{MODEL_NAME}_final.pth")